# Tokenizer Tutorial <br>

This notebook will serve as a tutorial for how to use the `tokenizer` class in the Lexos API to divide texts into "tokens". A token is a specific part of a text that is used for computational analysis. Most of the time, tokens are words, but punctuation, digits, and whitespace can also be tokens. The point of this class is to divide a text or set of text into tokens, which can then be used in further analysis.

## Tokenizer Background <br>

The `tokenizer` class uses language models in order to tokenize texts, as language models are far better at tokenization across a wide variety of languages and dialects than a simple text splitter. Using language models is more intsensive than using a basic Python tool (such as .split()), but is far more comprehensive and more likley to correctly tokenize texts due to language models' being trained on different languages' intricises. <br> <br>
The `tokenizer` class uses the python library spaCy as the backbone of its' tokenization method. `tokenizer` uses spaCy language models to tokenize texts, and the return type of each tokenization call is a spaCy document (or doc). Each spaCy doc contains the original text(s), a list of tokens, and attributes of each token, including `is_punct` and `is_digit`. Depending on the model used, the doc may contain more or less information. For example, using the spaCy model `"en_web_core_sm"` on an english text tags each token with the part of speech of every word. <br> <br>
Using language models does present some tradeoffs. Becuase language models are trained to specific languages, they can be "opinionated" and overfit the data. Similarly, if no language model exists for the language a text is written in, then the tokenization process will not be sufficent. In an effort to minimize these challanges, tokenizer uses the multi-language `"xx_sent_ud_sm"` model by default, which while not providing as much detail as other models, is a good basis for analysis across a number of different languages. There are many different spaCy models that can be used that span many different languages, all of which are compatible with the tokenizer class given their installation locally.

## Import `tokenizer` <br>
First, we need to import the `tokenizer` class from the Lexos module.

In [ ]:
from lexos import tokenizer
from lexos.tokenizer import Tokenizer

## Load Data <br>
Next. we'll load some data to tokenize. To do this, we'll use the `loader` module to load in a text file from Github. The sample file we'll be using is a small portion of "Pride and Prejudice" by Jane Austen. 

In [ ]:
from lexos.io import loader
loader = loader.Loader()
loader.load(["https://raw.githubusercontent.com/scottkleinman/lexos/refs/heads/main/tests/test_data/txt/Austen_Pride_sm.txt"])
text = loader.texts[0]
text

## Tokenizing a Single Text  <br>
Once we load some data, then we can tokenize the text. The `tokenizer` class uses the function `make_doc()` to tokenize a text, where the text is fed to that returns a spaCy doc. You can also call the `tokenizer` object directly as a function, which will then use `make_doc()` in the same way that a normal call to `make_doc()` would function. Either call will return a spaCy doc object. <br><br>
After the text is tokenized and we have a spaCy doc, we can print out the first 50 tokens by using a for loop. Notice how some tokens are punctuation or line breaks - this can be remedied by either scrubbing the text before tokenization, or by filtering the tokens post-tokenization. <br><br>
You can access the original text by referencing `doc.text`, and by using the bracket operators, you can access a substring of the original text.

In [ ]:
tokenizer_def = Tokenizer()
doc = tokenizer_def.make_doc(text)

print("\nTokens:")
for token in doc[0:38]:
    print(f"<{token.text}>")

# Alernatively, you can call the Tokenizer object directly:
doc = tokenizer_def(text)

# Access the original text:
org_text = doc.text[0:100]

## Tokenizing Multiple Texts <br>
Tokenizer also contains the function `make_docs()`, which creates multiple spaCy docs from a list of texts. For this example, we'll split our subsection of "Pride and Prejudice" into a further two subsections. Similarly to `make_doc()`, we can also call the tokenizer object directly and pass the list of texts to create multiple spaCy docs. <br><br>
Although not used in this example, it should be noted that `loader.texts` is a list itself, so a common practice would be to pass `loader.texts` into a call to `make_docs()`.

In [ ]:
text_sub1 = text[0:100]
text_sub2 = text[100:200]
text_list = [text_sub1, text_sub2]
docs = list(tokenizer_def.make_docs(text_list))

#Alternatively, you can call the Tokenizer object directly:
docs = list(tokenizer_def(text_list))

## Selecting a Model <br>
As mentioned previously, you can select the model that tokenizer uses in order to get more information from or text, or to better fit the language the text is in. In order to do this, you can use the `model` parameter in the `make_doc()` function to override the default model. For this example, we'll use the `'en_web_core_sm'` model, since "Pride and Prejudice" is written in english. This model tags parts of speech to each token, as shown below.

In [ ]:
tokenizer_en = Tokenizer(model="en_core_web_sm")
doc = tokenizer_en.make_doc(text)
print("\nTokens with parts of speech:")
for token in doc[0:50]:
    print(f"<{token.text}> : {token.pos_}")

## Adding/Removing Stop Words
Stop words are words that are to be excluded from the list of tokens. These are generally words like "the" and "and", but can be anything that serves the purpose of the desired analysis. <br> <br>
To add stopwords to a `tokenizer` instance, use the `add_stopwords()` function to pass a list of words that will act as stopwords before tokenizing. <br><br>
Some models, such as `"en_web_core_sm"`, have built-in stopwords. To remove these, or any other stop words, use the `remove_stopwords()` function before tokenizing.

In [ ]:
test_text = "This is a test of stopwords in the tokenizer class."

# Adding stopwords
tokenizer_def.add_stopwords(["is", "the", "of"])
stop_doc = tokenizer_def.make_doc(test_text)
for token in stop_doc[0:50]:
    print(f"Token: {token.text:<12}    Stopword: {token.is_stop}")
print("\n================================\n")

# Removing stopwords
tokenizer_def.remove_stopwords(["is", "the", "of"])
stop_doc = tokenizer_def.make_doc(test_text)
for token in stop_doc[0:50]:
    print(f"Token: {token.text:<12}    Stopword: {token.is_stop}")

## Filtering Docs <br>
To remove unwanted tokens from a doc (such as stopwords, punctuation, digits, etc.), you can use the attributes of each token to create new texts that filter out what you're trying to remove. After creating a new text, run `make_doc()` on the filtered text to create a filtered doc. <br><br>
Note: Besides the `text` and all attributes that begin with `is_`, in order to get the value of attributes, you will need to add a trailing underscore to the attribute. For example, calling the `pos` attribute gives the numberic value of the part of speech of the token, while the `pos_` attribute gives the actual part of speech

In [ ]:
# Use a list comprehension to create a list of filtered tokens forms
filtered_tokens = [token.text for token in doc if not token.is_stop and not token.is_punct and not token.is_space]

# Convert the list to a space-separated string and make a new doc
filtered_text = " ".join(filtered_tokens)
filtered_doc = tokenizer_def.make_doc(filtered_text)

for token in filtered_doc[0:38]:
    print(f"<{token.text}>")

## Adjusting spaCy Pipelines <br>
Sometimes, the size of a text and the amount of information being collected by a language model can severly slow down the processing speed of a tokenization call. If one would like to disable a component of a model in order to speed up processing speed, then one can use the `remove_extension()` method to disable a component of the model. <br><br>
Similarly, if one would like to add or re-enable a component of a model, then they can use the `add_extension()` method to do just that.

In [ ]:
tokenizer_en.remove_extension("tagger")

tokenizer_en.add_extension("tagger", default="default_value")

## Simple Tokenizers <br>
Along with the language model based tokenizer, the `tokenizer` class also contains two simple tokenizers: `SliceTokenizer` and `WhitespaceTokenizer`. <br><br>
`SliceTokenizer` slices the text into tokens of n characters. The constructor takes two arguments: `n`, which is the number of characters that each token will be, and `drop_ws`, a modifier that controls whether to drop whitespace or keep it.

In [ ]:
from lexos.tokenizer import SliceTokenizer
test_text = "Cut me up into tiny pieces!"
slicer = SliceTokenizer(n = 4, drop_ws=True)
slices = slicer(test_text)
print(slices)

`WhitespaceTokenizer` simply slices a text into tokens on whitespace, similarly to the built-in `split()` method. 

In [ ]:
from lexos.tokenizer import WhitespaceTokenizer
test_text = "Split me up by whitespace!"
neatSlicer = WhitespaceTokenizer()
slices = neatSlicer(test_text)
print(slices)

## Ngrams <br>

The `tokenizer` class includes a subclass called `ngrams`, which allows you to tokenize a text into segements of 1 or more units of a text. Usually, n-grams are made up of words, but they can also be made up of characters, sentences, etc. <br><br>

In [ ]:
from lexos.tokenizer.ngrams import Ngrams
ngrams_generator = Ngrams(n=3, tokenizer=tokenizer_en)

### Generating Word Ngrams <br>
To generate ngrams from a string of text, use the `from_text()`. For a spaCy doc, use the `from_doc()` method. For a list of tokens, use the `from_tokens()` method. <br><br>
To select the number of units in the ngrams, use the parameter `n`. There are plenty of parameters that can be utilized to filter the text as well, such as `drop_ws` and `filter_stop`. The `Ngrams` methods also have a parameter `tokenizer` that allows you to select which tokenizer to use to construct the ngrams. By default, the tokenizer used is `WhitespaceTokenizer`. Use the `output` parameter to select the kind of output that you would desire. You can select text, spans, or touples for `from_docs()` and `from_text()`, while only text and tuples are available outputs for `from_tokens()`. <br><br>
By default, `n` = 2 and `output` = text. 

In [ ]:
ngrams = ngrams_generator.from_text(text=text, output="text", n=3, tokenizer=tokenizer_def, drop_ws=True)
ngrams_list = list(ngrams)
for ngram in ngrams_list[0:10]:
    print(ngram)
print ("================================\n")

doc = tokenizer_def.make_doc(text)
ngrams = ngrams_generator.from_doc(doc=doc, output="tuples", n=3)
ngrams_list = list(ngrams)
for ngram in ngrams_list[0:10]:
    print(ngram)
print ("================================\n")

tokens_list = [token.text for token in doc]
ngrams = ngrams_generator.from_tokens(tokens=tokens_list, output="tuples", n=3, drop_ws=True)
ngrams_list = list(ngrams)
for ngram in ngrams_list[0:10]:
    print(ngram)

You can also use the methodd `from_texts()`, `from_docs()`, and `from_token_lists()` to generate ngrams from a list of texts, spaCy docs, or multiple lists of tokens. 

In [ ]:
ngrams = ngrams_generator.from_texts(texts=text_list, output="text", n=3, tokenizer=tokenizer_def, drop_ws=True)
for doc in ngrams:
    doc_list = list(doc)
    for ngram in doc_list[0:10]:
        print(ngram)
    print("------------------------")
print ("\n================================\n")

ngrams = ngrams_generator.from_docs(docs=docs, output="tuples", n=3)
for doc in ngrams:
    doc_list = list(doc)
    for ngram in doc_list[0:10]:
        print(ngram)
    print("------------------------")
print ("\n================================\n")

tokens_list_1 = [token.text for token in docs[0]]
tokens_list_2 = [token.text for token in docs[1]]
ngrams = ngrams_generator.from_token_lists([tokens_list_1, tokens_list_2], output="tuples", n=3, drop_ws=True)
for doc in ngrams:
    doc_list = list(doc)
    for ngram in doc_list[0:10]:
        print(ngram)
    print("------------------------")
print ("\n================================\n")

## Character Ngrams <br>
To get ngrams using characters, you use `SliceTokenizer` as the tokenizer in the ngrams method.

In [ ]:
ngrams = ngrams_generator.from_text(text=text, tokenizer=slicer, output="tuples", n=5)
ngrams_list = list(ngrams)
for ngram in ngrams_list[0:10]:
    print(ngram)